# Plot `met_forcing_rxn` simulations

Compare control and enhanced rock weathering simulations run with long-term mean, monthly, daily, and hourly meteorological forcing.

The notebook focuses on drainage and alkalinity export, final geochemical profiles, mean soil-water and gas profiles, and near-surface diffusive gas efflux.


In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.notebook import tqdm
import numpy as np

from byte_util.util import all_sites, all_forcing_types, calc_specific_discharge
from byte_util.met_forcing_plots import plot_cumulative_export, temperature_K

## Configuration


In [ ]:
sim_root = Path('../min3p_runs')
plot_folder = Path('./plots')

control_planes = [50, 100, 200, 300]
hourly_reference = 'hourly'
forcing_colors = dict(zip(all_forcing_types,
                          plt.colormaps['Set2'](np.linspace(0.05, 0.95, 4))))

## Drainage and alkalinity export

Specific discharge is calculated at `control_plane_cm` using the pressure head output and the model input file. Alkalinity is selected from the upstream cell at the same control plane.


In [ ]:
pdf_file = plot_folder / f'CumulativeAlkalinityExport.pdf'
replot = True

if replot:
    with PdfPages(pdf_file) as pdf:
        for control_plane in control_planes:
            # Add label for this control plane
            fig, ax = plt.subplots(figsize=(8, 5))
            ax.annotate(f'Control plane = {control_plane:d} cm', (0.5, 0.5), fontsize=32, ha='center', va='center')
            ax.xaxis.set_visible(False)
            ax.yaxis.set_visible(False)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            for site in tqdm(all_sites):
                fig, ax = plot_cumulative_export(site, control_plane_cm=control_plane)
                pdf.savefig(fig, bbox_inches='tight')
                plt.close(fig)

## Plot final profiles across all scenarios

Note that the "final" profile might not be at 10 years

In [ ]:
from byte_util.met_forcing_plots import plot_final_profiles, plot_final_erw_difference
replot = True

pdf_file = plot_folder / 'FinalERWProfiles.pdf'
if replot:
    with PdfPages(pdf_file) as pdf:
        for site in tqdm(all_sites):
            fig, ax = plot_final_profiles(site, ['h+1', 'Alk [eq/L]', 'mg+2', 'forst_vol'])
            pdf.savefig(fig, bbox_inches='tight')
            if site != 'HoustonBlack':
                plt.close(fig)

            fig, ax = plot_final_erw_difference(site, ['h+1', 'Alk [eq/L]', 'mg+2'])
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

In [ ]:
from byte_util.met_forcing_plots import plot_mean_spatial_gas_profiles, plot_mean_transient_gas_profiles
replot = True

pdf_file = plot_folder / 'MeanWaterContentCO2Profiles.pdf'
if replot:
    with PdfPages(pdf_file) as pdf:
        for site in tqdm(all_sites):
            fig, ax = plot_mean_spatial_gas_profiles(site)
            pdf.savefig(fig, bbox_inches='tight')
            plt.close(fig)

            fig, ax = plot_mean_transient_gas_profiles(site)
            pdf.savefig(fig, bbox_inches='tight')
            if site != 'HoustonBlack':
                plt.close(fig)

## Bar chart of feedstock dissolution after 10 years

In [ ]:
from byte_util.met_forcing_plots import plot_feedstock_dissolution

fig, ax = plot_feedstock_dissolution()